# OEC Demo Acceptance Index — current source checkout

This is a **read-only preflight/index notebook**. It does not start a model or install packages. Use it to confirm the checkout and GPU state, then use the model-specific notebooks and the corresponding OEC Actions acceptance run.

## Model notebooks and modes

- **Audio generation:** \`pipeline_audiocraft.ipynb\` (full legacy notebook), \`pipeline_audiogen_gradio.ipynb\` (AudioGen server client), \`pipeline_musicgen_gradio.ipynb\` (MusicGen server client), and \`pipeline_acestep_gradio.ipynb\` (ACE-Step text/music/lyrics controls).
- **Speech/ASR:** \`pipeline_kokoro_gradio.ipynb\`, \`pipeline_whisper_gradio.ipynb\`, and \`pipeline_qwen3_asr_gradio.ipynb\`. The acceptance harness chains Kokoro-generated WAV input into Whisper and Qwen3-ASR callbacks.
- **Images and 3D:** \`pipeline_flux_only.ipynb\`, \`pipeline_trellis_3d_only.ipynb\`, and \`pipeline_trellis_3d.ipynb\` (separate TRELLIS image-to-3D and FLUX text-to-image modes).
- **Video:** \`pipeline_ltx_video.ipynb\` supports text-to-video and image-to-video; \`pipeline_wan_video.ipynb\` runs the guarded full-code-cell video path.
- **Operational notebooks:** \`phase1_test_compute_node.ipynb\` is for compute-node diagnostics; \`phase1_test_login_node.ipynb\` is for the login/control plane; \`oec_deply_install.ipynb\` checks deployment state and has an explicit opt-in server launch; \`pipeline_verify_encrypted_hf.ipynb\` requires a real encrypted object and the correct private-key/decryptor context.

## Safe sequence

1. Run this notebook from the repository root and inspect the current branch/commit/GPU.
2. Check StorageManager status and the acceptance artifact for each model/mode.
3. Run one GPU-heavy model at a time. Start only its isolated, loopback-bound service after GPU and storage admission.
4. Validate the returned audio/video/image/GLB file, not just a successful Gradio launch.
5. Stop the service, then run lock-aware cache cleanup for the tested model. A live or remote/unknown runtime lock blocks deletion.

Historical Actions results are evidence for their own commit only; they are not proof for a later checkout. The canonical shared \`$HOME/OEC\` checkout and deployment/credential-specific notebooks must be validated in their own approved context.

In [ ]:
from pathlib import Path
import subprocess
ROOT = Path.cwd().resolve()
if not (ROOT / 'storage_manager.py').is_file():
    raise RuntimeError('Run this notebook from the OEC repository root.')
print('OEC_ROOT=', ROOT)
for command in (['git','branch','--show-current'], ['git','rev-parse','HEAD'], ['git','status','--short','--branch'], ['nvidia-smi']):
    result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=False)
    print('$', ' '.join(command), '\n', result.stdout or result.stderr)
print('Before inference, run the read-only StorageManager preflight:')
print('$HOME/Conda/condabin/conda run --no-capture-output -n oec_deploy python storage_manager.py --status')


## Acceptance expectations

The maintained acceptance workflow audits all notebook JSON and transformed code cells, then runs real model-server callbacks where the required runtime is available. It captures the generation report, artifact SHA-256 and decoded stream metadata, model-cache cleanup report, StorageManager status, and GPU state.

Modes which require a login-node control plane, a running deployed service, a remote encrypted object, or private-key credentials are statically checked here and must be exercised in that dedicated context. Do not treat a control-plane or credential-dependent mode as a live inference pass merely because this index notebook executed successfully.

## Safety reminders

- Keep model services bound to loopback; use SSH tunnels for remote access.
- Never upgrade Torch/Diffusers or install model stacks in the shared notebook kernel during inference.
- Never terminate an unknown PID or purge a cache with an active/remote/unknown runtime lock.
- Use fresh StorageManager admission for cache fills; preserve reserve files, source checkouts, datasets, and generated deliverables.
- Never write API tokens/private keys or decrypted private artifacts into notebook outputs.